# Projeto Deméter — portfólio de visão computacional

Projeto acadêmico em equipe. Nesta apresentação, o nome fictício Deméter substitui a identificação do projeto original. Minha responsabilidade foi o componente de visão computacional, incluindo preparação dos dados, treinamento, avaliação e entrega dos pesos para integração pela equipe.

Dois experimentos de detecção de plantas com YOLOv8n, com preparação de dados, treinamento e avaliação.

**Como usar**
- Abra no Google Colab e selecione um ambiente com GPU.
- O experimento 1 requer o Dataset.zip original, com datasets/weed_crop_cc0 em sua raiz. O arquivo não está incluído neste repositório.
- O experimento 2 pode ser iniciado em uma sessão própria a partir de sua seção. Os dados são baixados do Zenodo.
- Execute as células em ordem dentro do experimento escolhido. Baixe os resultados antes de encerrar o Colab.

**Sobre esta edição**
Os resultados textuais selecionados foram preservados da execução enviada pela autora. Imagens embutidas, widgets e logs extensos foram removidos para reduzir o arquivo. Os caminhos de pastas com data fixa foram substituídos pelas variáveis criadas durante a execução. Esta edição não foi reexecutada.

As métricas dos experimentos usam conjuntos de teste diferentes. A independência espacial dos conjuntos CornWeed não está comprovada.


## 1. Verificação da GPU

Utilizamos uma GPU para acelerar os cálculos do treinamento.
O comando abaixo mostra a GPU disponibilizada pelo Colab, sua memória
e os processos que estão utilizando o dispositivo.

Nesta sessão, foi identificada uma GPU NVIDIA Tesla T4.

In [ ]:
import torch

print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU disponível: True
GPU: Tesla T4


# Projeto Deméter — Detecção de plantas com YOLO

Este notebook apresenta a preparação dos dados, o treinamento e a avaliação
de um modelo YOLO para detectar duas classes em imagens agrícolas:

- **crop (classe 0):** planta cultivada.
- **weed (classe 1):** planta daninha.

O dataset contém 1.300 imagens com anotações, divididas em:
- **Treino:** 1.040 imagens para o aprendizado do modelo.
- **Validação:** 130 imagens para acompanhar o desempenho durante o treinamento.
- **Teste:** 130 imagens reservadas para a avaliação final.

O treinamento será executado no Google Colab com GPU.
Ao final, o modelo será integrado à aplicação Projeto Deméter.

## 2. Envio do dataset

Enviamos o arquivo Dataset.zip para o ambiente do Colab.
Ele contém as imagens e as anotações de treino, validação e teste.
O arquivo será extraído no servidor para preparar o treinamento.

In [ ]:
from google.colab import files

uploaded = files.upload()

## 3. Extração e conferência do dataset

Extraímos o dataset para /content, no servidor do Colab.
A estrutura original de treino, validação e teste será mantida.

Em seguida, contamos as imagens e os arquivos de anotação de cada conjunto.
Cada imagem deve ter uma anotação correspondente em formato TXT.

In [ ]:
from pathlib import Path
import zipfile

# Localiza o ZIP enviado na etapa anterior
arquivo_zip = next(
    nome for nome in uploaded
    if nome.lower().endswith(".zip")
)

# Extrai os arquivos no Colab
with zipfile.ZipFile(arquivo_zip, "r") as zip_ref:
    zip_ref.extractall("/content")

dataset = Path("/content/datasets/weed_crop_cc0")

# Conta imagens e anotações
for conjunto in ["train", "val", "test"]:
    imagens = [
        p for p in (dataset / "images" / conjunto).iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    anotacoes = list((dataset / "labels" / conjunto).glob("*.txt"))

    print(
        f"{conjunto}: {len(imagens)} imagens | "
        f"{len(anotacoes)} anotações"
    )

print("\nConfiguração do dataset:")
print((dataset / "data.yaml").read_text())

train: 1040 imagens | 1040 anotações
val: 130 imagens | 130 anotações
test: 130 imagens | 130 anotações

Configuração do dataset:
train: images/train
val: images/val
test: images/test
names:
  0: crop
  1: weed



## 4. Instalação da biblioteca de visão computacional

Utilizamos a biblioteca Ultralytics, conforme o script train_vision.py
do Projeto Deméter.

O modelo base será o YOLOv8n, uma versão compacta do YOLO.
Aplicaremos transferência de aprendizado: adaptaremos pesos
pré-treinados para detectar as classes crop e weed do nosso dataset.

A instalação abaixo ocorre no ambiente do Colab.

In [ ]:
%pip install -q ultralytics==8.4.190


## 5. Configuração dos caminhos do dataset

Criamos uma configuração específica para o Colab, indicando o caminho
completo do dataset e mantendo a divisão original dos dados e das classes.

O arquivo data.yaml original será preservado.

In [ ]:
from pathlib import Path
import yaml

dataset = Path("/content/datasets/weed_crop_cc0")

# Lê a configuração original
config = yaml.safe_load(
    (dataset / "data.yaml").read_text()
)

# Define onde o dataset está no Colab
config["path"] = str(dataset)

# Salva uma configuração separada
data_colab = dataset / "data_colab.yaml"

data_colab.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8"
)

print(data_colab.read_text())

train: images/train
val: images/val
test: images/test
names:
  0: crop
  1: weed
path: /content/datasets/weed_crop_cc0



## 6. Visualização das imagens e anotações

Exibimos quatro imagens do conjunto de treino com as caixas
delimitadoras fornecidas no dataset.

Cada anotação informa a classe e a posição de uma planta.
As caixas verdes representam crop (planta cultivada) e as
vermelhas representam weed (planta daninha).

Esta inspeção permite conferir exemplos antes do treinamento.

In [ ]:
from pathlib import Path
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

dataset = Path("/content/datasets/weed_crop_cc0")

imagens = sorted(
    p for p in (dataset / "images/train").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

amostra = random.Random(42).sample(imagens, 4)
classes = {0: "crop", 1: "weed"}
cores = {0: "lime", 1: "red"}

fig, eixos = plt.subplots(2, 2, figsize=(14, 10))

for eixo, caminho in zip(eixos.flat, amostra):
    imagem = Image.open(caminho).convert("RGB")
    largura, altura = imagem.size
    eixo.imshow(imagem)

    anotacao = dataset / "labels/train" / f"{caminho.stem}.txt"

    for linha in anotacao.read_text().splitlines():
        if not linha.strip():
            continue

        classe, x, y, w, h = map(float, linha.split())
        classe = int(classe)

        # Converte as coordenadas normalizadas para pixels
        esquerda = (x - w / 2) * largura
        topo = (y - h / 2) * altura

        caixa = patches.Rectangle(
            (esquerda, topo),
            w * largura,
            h * altura,
            linewidth=2,
            edgecolor=cores[classe],
            facecolor="none"
        )
        eixo.add_patch(caixa)
        eixo.text(
            esquerda, topo, classes[classe],
            color="white",
            backgroundcolor=cores[classe]
        )

    eixo.set_title(caminho.name)
    eixo.axis("off")

plt.tight_layout()
plt.show()

## 7. Distribuição das classes

Contamos quantas plantas estão anotadas como crop e weed em cada
conjunto. Uma imagem pode conter várias plantas, portanto a quantidade
de objetos anotados pode ser maior que a quantidade de imagens.

In [ ]:
from collections import Counter

for conjunto in ["train", "val", "test"]:
    contagem = Counter()

    for arquivo in (dataset / "labels" / conjunto).glob("*.txt"):
        for linha in arquivo.read_text().splitlines():
            if linha.strip():
                classe = int(linha.split()[0])
                contagem[classe] += 1

    print(
        f"{conjunto}: "
        f"{contagem[0]} plantas crop | "
        f"{contagem[1]} plantas weed"
    )

train: 946 plantas crop | 673 plantas weed
val: 140 plantas crop | 84 plantas weed
test: 126 plantas crop | 103 plantas weed


## 8. Armazenamento dos resultados

Os pesos e resultados serão salvos temporariamente no Colab.
Após o treinamento, serão baixados para o computador.

É necessário baixar os arquivos antes de encerrar ou perder
o ambiente de execução.

In [ ]:
from pathlib import Path
import torch

pasta_resultados = Path("/content/resultados_Projeto Deméter")
pasta_resultados.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "GPU indisponível."
assert Path(
    "/content/datasets/weed_crop_cc0/data_colab.yaml"
).is_file(), "Configuração do dataset não encontrada."

print("Tudo pronto para treinar!")
print("GPU:", torch.cuda.get_device_name(0))

Tudo pronto para treinar!
GPU: Tesla T4


## 9. Treinamento do YOLOv8n

Utilizamos o modelo pré-treinado YOLOv8n para aprender a detectar
plantas cultivadas (crop) e plantas daninhas (weed).

O treinamento utiliza 20 épocas, imagens de tamanho 512 e lotes
de 8 imagens, conforme os parâmetros iniciais do script do projeto.

Os cálculos serão executados na GPU Tesla T4.
O conjunto de validação acompanha o desempenho durante o treino;
o conjunto de teste fica reservado para a avaliação final.

In [ ]:
from ultralytics import YOLO

# Carrega o modelo pré-treinado
modelo = YOLO("yolov8n.pt")

# Inicia o treinamento na GPU
resultados = modelo.train(
    data="/content/datasets/weed_crop_cc0/data_colab.yaml",
    epochs=20,
    imgsz=512,
    batch=8,
    device=0,
    workers=2,
    project="/content/resultados_Projeto Deméter",
    name="crop_weed_20ep",
    exist_ok=False,
    seed=42
)

Validating /content/resultados_Projeto Deméter/crop_weed_20ep/weights/best.pt...
Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.0it/s 2.2s
                   all        130        224      0.801      0.701      0.779       0.51
                  crop         63        140      0.738      0.664      0.748      0.517
                  weed         68         84      0.863      0.738       0.81      0.504
Speed: 1.9ms preprocess, 5.9ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to /content/resultados_Projeto Deméter/crop_weed_20ep


In [ ]:
import shutil
from google.colab import files

arquivo = shutil.make_archive(
    "/content/Demeter_treino_20ep",
    "zip",
    "/content/resultados_Projeto Deméter/crop_weed_20ep"
)

files.download(arquivo)

### Resultado do treinamento inicial

O treinamento de 20 épocas foi concluído utilizando a GPU Tesla T4.

O melhor modelo apresentou, no conjunto de validação:
- Precisão: 80,1%.
- Recall: 70,1%.
- mAP@50: 77,9%.
- mAP@50–95: 51,0%.

Esses valores descrevem o desempenho na validação e não equivalem
a uma porcentagem geral de acerto. A avaliação no conjunto de teste
será realizada separadamente.

## 10. Avaliação no conjunto de teste

Carregamos o best.pt, selecionado pelo desempenho na validação,
e avaliamos nas 130 imagens reservadas para teste.

Esta etapa não altera os pesos do modelo. Calculamos precisão,
recall, mAP@50 e mAP@50–95, além de gerar gráficos e exemplos
de detecções para analisar o desempenho.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import json

melhor_modelo = YOLO(
    "/content/resultados_Projeto Deméter/crop_weed_20ep/weights/best.pt"
)

print("Classes do modelo:", melhor_modelo.names)

metricas_teste = melhor_modelo.val(
    data="/content/datasets/weed_crop_cc0/data_colab.yaml",
    split="test",
    imgsz=512,
    batch=8,
    device=0,
    workers=2,
    plots=True,
    project="/content/resultados_Projeto Deméter",
    name="avaliacao_teste",
    exist_ok=False
)

# Salva as métricas numéricas para a documentação
pasta_teste = Path(metricas_teste.save_dir)

resumo = {
    chave: float(valor)
    for chave, valor in metricas_teste.results_dict.items()
}

(pasta_teste / "metricas_teste.json").write_text(
    json.dumps(resumo, indent=2),
    encoding="utf-8"
)

print("\nMétricas do teste:")
print(json.dumps(resumo, indent=2))

Classes do modelo: {0: 'crop', 1: 'weed'}
Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1914.7±387.6 MB/s, size: 63.8 KB)
val: Scanning /content/datasets/weed_crop_cc0/labels/test... 130 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 130/130 1.8Kit/s 0.1s
val: New cache created: /content/datasets/weed_crop_cc0/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 8.9it/s 1.9s
                   all        130        229      0.808      0.782      0.848      0.551
                  crop         59        126      0.733      0.833      0.863      0.578
                  weed         71        103      0.883      0.731      0.832      0.524
Speed: 1.8ms preprocess, 5.1ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /con

### Resultados no conjunto de teste

O modelo foi avaliado em 130 imagens, contendo 229 plantas anotadas.
Não foram identificados arquivos corrompidos.

Os resultados gerais foram:
- Precisão: 80,8%.
- Recall: 78,2%.
- mAP@50: 84,8%.
- mAP@50–95: 55,1%.

A classe weed apresentou precisão de 88,3% e recall de 73,1%,
indicando que parte das plantas daninhas não foi detectada.
Os resultados se referem ao conjunto de teste utilizado e não
garantem o mesmo desempenho em outras condições agrícolas.

## 11. Exemplos de detecções

Executamos o modelo treinado nas imagens de teste e salvamos
as caixas previstas, as classes e os níveis de confiança.

Utilizamos confiança mínima de 0,25 para esta visualização.


In [ ]:
for resultado in melhor_modelo.predict(
    source="/content/datasets/weed_crop_cc0/images/test",
    imgsz=512,
    conf=0.25,
    device=0,
    save=True,
    stream=True,
    project="/content/resultados_Projeto Deméter",
    name="deteccoes_teste",
    exist_ok=False
):
    pass

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random

pasta = Path("/content/resultados_Projeto Deméter/deteccoes_teste")

imagens = sorted(
    p for p in pasta.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

amostra = random.Random(42).sample(imagens, 6)

fig, eixos = plt.subplots(2, 3, figsize=(16, 10))

for eixo, arquivo in zip(eixos.flat, amostra):
    eixo.imshow(Image.open(arquivo))
    eixo.set_title(arquivo.name)
    eixo.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import shutil
from google.colab import files

# Copia o melhor modelo com o nome esperado pelo Projeto Deméter
shutil.copy2(
    "/content/resultados_Projeto Deméter/crop_weed_20ep/weights/best.pt",
    "/content/resultados_Projeto Deméter/weed_crop_best.pt"
)

arquivo_zip = shutil.make_archive(
    "/content/Demeter_YOLO_completo",
    "zip",
    "/content/resultados_Projeto Deméter"
)

files.download(arquivo_zip)

# Experimento 2 — Inspeção do dataset CornWeed

Vamos verificar o dataset de milho e plantas daninhas antes de preparar um novo treinamento. O primeiro experimento permanece preservado acima.

In [ ]:
%pip install -q ultralytics==8.4.190 scipy matplotlib pillow tqdm requests pyyaml

from pathlib import Path
import shutil

pasta = Path("/content/cornweed")
pasta.mkdir(parents=True, exist_ok=True)

livre = shutil.disk_usage("/content").free / 1024**3
print(f"Espaço livre: {livre:.1f} GB")

## Download do CornWeed

O arquivo será baixado diretamente no armazenamento temporário do Colab para inspeção, sem alterar o dataset ou o modelo do primeiro experimento.

In [ ]:
!wget -c --progress=bar:force:noscroll \
  "https://zenodo.org/records/7961764/files/maize_dataset.zip?download=1" \
  -O "/content/cornweed/maize_dataset.zip"

## Verificação da integridade do ZIP

Calculamos o MD5 do arquivo baixado e comparamos com o checksum publicado na fonte. Esta etapa não extrai os dados nem inicia treinamento.

In [ ]:
from pathlib import Path
import hashlib

zip_cornweed = Path("/content/cornweed/maize_dataset.zip")
md5_esperado = "9b409e835b51c0f7b01524dcb1dfeda6"

md5 = hashlib.md5()

with zip_cornweed.open("rb") as arquivo:
    for bloco in iter(lambda: arquivo.read(8 * 1024 * 1024), b""):
        md5.update(bloco)

md5_obtido = md5.hexdigest()
print("MD5 obtido:", md5_obtido)

assert md5_obtido == md5_esperado, "Checksum diferente: precisamos investigar."
print("Integridade confirmada!")

MD5 obtido: 9b409e835b51c0f7b01524dcb1dfeda6
Integridade confirmada!


## Inspeção da estrutura do CornWeed

Listamos os arquivos de configuração, documentação e anotações para identificar as classes, divisões e condições de uso antes de preparar o treinamento.

In [ ]:
import zipfile

with zipfile.ZipFile(zip_cornweed) as arquivo:
    membros = arquivo.infolist()
    print("Total de entradas no ZIP:", len(membros))

    for membro in membros:
        nome = membro.filename.lower()
        if (
            nome.endswith((".json", ".yaml", ".yml", ".names", ".data"))
            or any(palavra in nome for palavra in
                   ("license", "licence", "readme", "classes", "split"))
        ):
            print(f"{membro.filename} | {membro.file_size:,} bytes")

## Classes e divisões originais

Lemos a configuração, a documentação e as categorias COCO, preservando os rótulos originais. Nenhuma classe será renomeada nesta etapa.

In [ ]:
import json
from collections import Counter

with zipfile.ZipFile(zip_cornweed) as arquivo:
    for nome in [
        "maize_dataset/data.yaml",
        "maize_dataset/README.txt",
    ]:
        print(f"\n--- {nome} ---")
        print(arquivo.read(nome).decode("utf-8-sig", errors="replace"))

    for divisao in ["train", "valid"]:
        nome = f"maize_dataset/coco_annotations/{divisao}_2022.json"

        with arquivo.open(nome) as entrada:
            dados = json.load(entrada)

        print(f"\n--- COCO: {divisao} ---")
        print("Categorias:", dados.get("categories"))
        print("Licenças:", dados.get("licenses"))
        print("Informações:", dados.get("info"))
        print("Imagens:", len(dados.get("images", [])))
        print("Instâncias por categoria:", dict(Counter(
            item["category_id"]
            for item in dados.get("annotations", [])
        )))

## Inspeção dos rótulos e listas de imagens

Verificamos exemplos das anotações YOLO e os arquivos que definem as divisões originais, antes de modificar classes ou preparar novos conjuntos.

In [ ]:
with zipfile.ZipFile(zip_cornweed) as arquivo:
    nomes = arquivo.namelist()
    textos = [
        nome for nome in nomes
        if nome.lower().endswith(".txt")
        and not nome.endswith("/")
    ]

    listas = [
        nome for nome in textos
        if nome.rsplit("/", 1)[-1].lower()
        in {"train.txt", "valid.txt", "val.txt", "test.txt"}
    ]

    for nome in listas:
        linhas = arquivo.read(nome).decode(
            "utf-8-sig", errors="replace"
        ).splitlines()
        linhas = [linha.strip() for linha in linhas if linha.strip()]
        print(f"\nLISTA: {nome} | {len(linhas)} entradas")
        print("\n".join(linhas[:5]))

    rotulos = [
        nome for nome in textos
        if nome not in listas
        and nome.rsplit("/", 1)[-1].lower() != "readme.txt"
    ]

    print("\nQuantidade de outros arquivos TXT:", len(rotulos))

    for nome in rotulos[:3]:
        print(f"\nEXEMPLO: {nome}")
        conteudo = arquivo.read(nome).decode(
            "utf-8-sig", errors="replace"
        )
        print("\n".join(conteudo.splitlines()[:8]))

## Verificação das sequências de aquisição

Agrupamos os nomes das imagens pelo prefixo anterior ao número do quadro, para identificar sequências compartilhadas entre treino e validação. Essa verificação não substitui a análise de similaridade visual.

In [ ]:
from collections import Counter
from pathlib import PurePosixPath
import re

grupos_por_divisao = {}
arquivos_por_divisao = {}

with zipfile.ZipFile(zip_cornweed) as arquivo:
    for divisao in ["train", "valid"]:
        caminho = f"maize_dataset/{divisao}.txt"
        linhas = arquivo.read(caminho).decode("utf-8-sig").splitlines()
        nomes = [
            PurePosixPath(linha.strip()).name
            for linha in linhas if linha.strip()
        ]

        grupos = Counter(
            re.sub(r"_\d+$", "", PurePosixPath(nome).stem)
            for nome in nomes
        )

        grupos_por_divisao[divisao] = grupos
        arquivos_por_divisao[divisao] = set(nomes)

        print(f"\n{divisao.upper()}:")
        for grupo, quantidade in sorted(grupos.items()):
            print(f"  {grupo}: {quantidade} imagens")

print("\nNomes presentes nas duas divisões:",
      len(arquivos_por_divisao["train"] &
          arquivos_por_divisao["valid"]))

print("Grupos presentes nas duas divisões:",
      sorted(set(grupos_por_divisao["train"]) &
             set(grupos_por_divisao["valid"])))


TRAIN:
  Langsenkamp_210722_bonirob_color: 882 imagens
  Langsenkamp_210727_bonirob_color: 500 imagens
  Langsenkamp_210727_color: 1480 imagens

VALID:
  Langsenkamp_210722_bonirob_color: 172 imagens
  Langsenkamp_210727_bonirob_color: 171 imagens
  Langsenkamp_210727_color: 369 imagens

Nomes presentes nas duas divisões: 0
Grupos presentes nas duas divisões: ['Langsenkamp_210722_bonirob_color', 'Langsenkamp_210727_bonirob_color', 'Langsenkamp_210727_color']


## Inspeção visual das anotações

Exibimos uma amostra de cada sequência com suas caixas originais: 0 = Weeds e 1 = Maize. As classes ainda não foram convertidas.

In [ ]:
import io
import random
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

rng = random.Random(42)

with zipfile.ZipFile(zip_cornweed) as arquivo:
    membros = set(arquivo.namelist())
    imagens = sorted(
        nome for nome in membros
        if nome.startswith("maize_dataset/data/")
        and nome.lower().endswith(".png")
    )

    grupos = {}
    for nome in imagens:
        grupo = re.sub(r"_\d+$", "", PurePosixPath(nome).stem)
        grupos.setdefault(grupo, []).append(nome)

    fig, eixos = plt.subplots(
        len(grupos), 1, figsize=(14, 7 * len(grupos)),
        squeeze=False
    )

    for eixo, (grupo, candidatos) in zip(eixos[:, 0], sorted(grupos.items())):
        nome = rng.choice(candidatos)
        imagem = Image.open(io.BytesIO(arquivo.read(nome))).convert("RGB")
        desenho = ImageDraw.Draw(imagem)
        largura, altura = imagem.size

        rotulo = str(PurePosixPath(nome).with_suffix(".txt"))
        assert rotulo in membros, f"Rótulo ausente: {rotulo}"

        for linha in arquivo.read(rotulo).decode("utf-8-sig").splitlines():
            if not linha.strip():
                continue
            classe, xc, yc, w, h = map(float, linha.split())
            caixa = (
                (xc - w / 2) * largura,
                (yc - h / 2) * altura,
                (xc + w / 2) * largura,
                (yc + h / 2) * altura,
            )
            cor = "red" if int(classe) == 0 else "lime"
            desenho.rectangle(caixa, outline=cor, width=2)

        eixo.imshow(imagem)
        eixo.set_title(
            f"{PurePosixPath(nome).name}\nVermelho: Weeds | Verde: Maize"
        )
        eixo.axis("off")

    plt.tight_layout()
    plt.show()

## Auditoria das anotações YOLO

Verificamos pares imagem/rótulo, IDs das classes e coordenadas das caixas. Os arquivos originais permanecem inalterados.

In [ ]:
import math
from collections import Counter

contagens = Counter()
problemas = []
vazios = 0

with zipfile.ZipFile(zip_cornweed) as arquivo:
    membros = set(arquivo.namelist())
    rotulos = sorted(
        nome for nome in membros
        if nome.startswith("maize_dataset/data/")
        and nome.lower().endswith(".txt")
    )

    for nome in rotulos:
        imagem = str(PurePosixPath(nome).with_suffix(".png"))
        if imagem not in membros:
            problemas.append((nome, "imagem PNG ausente"))

        linhas = arquivo.read(nome).decode("utf-8-sig").splitlines()
        linhas = [linha for linha in linhas if linha.strip()]
        if not linhas:
            vazios += 1

        for numero, linha in enumerate(linhas, 1):
            try:
                valores = list(map(float, linha.split()))
                if len(valores) != 5 or not all(map(math.isfinite, valores)):
                    raise ValueError("esperados 5 números finitos")

                classe, x, y, w, h = valores
                if classe not in (0, 1):
                    raise ValueError(f"classe inesperada: {classe}")
                if not (0 <= x <= 1 and 0 <= y <= 1
                        and 0 < w <= 1 and 0 < h <= 1):
                    raise ValueError("coordenadas fora do intervalo")

                contagens[int(classe)] += 1

                margem = 0.00001
                if (x - w/2 < -margem or y - h/2 < -margem
                        or x + w/2 > 1 + margem
                        or y + h/2 > 1 + margem):
                    problemas.append((nome, numero, "caixa ultrapassa a imagem"))

            except ValueError as erro:
                problemas.append((nome, numero, str(erro)))

    imagens = [
        nome for nome in membros
        if nome.startswith("maize_dataset/data/")
        and nome.lower().endswith(".png")
    ]
    sem_rotulo = [
        nome for nome in imagens
        if str(PurePosixPath(nome).with_suffix(".txt")) not in membros
    ]

print("Imagens em data/:", len(imagens))
print("Rótulos em data/:", len(rotulos))
print("Rótulos vazios:", vazios)
print("Instâncias por classe original:", dict(contagens))
print("Imagens sem rótulo:", len(sem_rotulo))
print("Problemas encontrados:", len(problemas))
print("Exemplos:", problemas[:10])

Imagens em data/: 3574
Rótulos em data/: 3574
Rótulos vazios: 0
Instâncias por classe original: {1: 23985, 0: 257740}
Imagens sem rótulo: 0
Problemas encontrados: 0
Exemplos: []


In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
import re
import zipfile

caminho_zip = Path("/content/cornweed/maize_dataset.zip")
assert caminho_zip.is_file(), f"ZIP não encontrado: {caminho_zip}"

pasta_dados = PurePosixPath("maize_dataset/data")
extensoes = {".png", ".jpg", ".jpeg"}
sequencias = defaultdict(list)

with zipfile.ZipFile(caminho_zip) as arquivo:
    arquivos = [
        item for item in arquivo.infolist()
        if not item.is_dir()
    ]

    nomes = [item.filename for item in arquivos]
    repetidos = [
        nome for nome, quantidade in Counter(nomes).items()
        if quantidade > 1
    ]
    assert not repetidos, f"Entradas repetidas: {repetidos[:10]}"

    dados = [
        item for item in arquivos
        if PurePosixPath(item.filename).parent == pasta_dados
    ]

    imagens = [
        item for item in dados
        if PurePosixPath(item.filename).suffix.lower() in extensoes
    ]
    rotulos = [
        item for item in dados
        if PurePosixPath(item.filename).suffix.lower() == ".txt"
    ]

    imagens_por_nome = defaultdict(list)
    rotulos_por_nome = defaultdict(list)

    for item in imagens:
        imagens_por_nome[PurePosixPath(item.filename).stem].append(
            item.filename
        )

    for item in rotulos:
        rotulos_por_nome[PurePosixPath(item.filename).stem].append(
            item.filename
        )

    assert len(imagens) == 3574, f"Imagens encontradas: {len(imagens)}"
    assert len(rotulos) == 3574, f"Rótulos encontrados: {len(rotulos)}"
    assert all(len(lista) == 1 for lista in imagens_por_nome.values())
    assert all(len(lista) == 1 for lista in rotulos_por_nome.values())
    assert imagens_por_nome.keys() == rotulos_por_nome.keys(), \
        "Há imagens ou rótulos sem par."

    print("Pares confirmados:", len(imagens))

    nao_interpretados = []

    for item in imagens:
        nome = PurePosixPath(item.filename).stem
        resultado = re.fullmatch(r"(.+)_(\d+)", nome)

        if resultado is None:
            nao_interpretados.append(nome)
            continue

        prefixo, numero = resultado.groups()
        sequencias[prefixo].append(int(numero))

    print("\nSEQUÊNCIAS NOMINAIS")
    print("Os nomes não comprovam independência espacial.")

    for prefixo, numeros in sorted(sequencias.items()):
        ordenados = sorted(set(numeros))
        passos = [
            seguinte - anterior
            for anterior, seguinte in zip(ordenados, ordenados[1:])
        ]
        lacunas = sorted(
            [
                (anterior, seguinte, seguinte - anterior)
                for anterior, seguinte in zip(ordenados, ordenados[1:])
                if seguinte - anterior > 1
            ],
            key=lambda item: item[2],
            reverse=True,
        )[:10]

        print(f"\nPrefixo: {prefixo}")
        print("Quantidade:", len(numeros))
        print("Primeiro e último quadro:", ordenados[0], ordenados[-1])
        print("Passos mais comuns:", Counter(passos).most_common(8))
        print("Maiores lacunas:", lacunas)

    print("\nNomes não interpretados:", nao_interpretados[:20])

    extensoes_metadados = {
        ".csv", ".tsv", ".json", ".yaml", ".yml", ".xml",
        ".gpx", ".kml", ".geojson", ".bag", ".db3",
        ".mcap", ".log", ".md", ".txt",
    }
    palavras = {
        "readme", "license", "licence", "metadata", "gps",
        "gnss", "trajectory", "timestamp", "calib", "camera",
        "sequence", "acquisition",
    }

    candidatos = []

    for item in arquivos:
        caminho = PurePosixPath(item.filename)

        if "testing" in {parte.lower() for parte in caminho.parts}:
            continue

        if (
            caminho.parent == pasta_dados
            and caminho.suffix.lower() in extensoes | {".txt"}
        ):
            continue

        if (
            caminho.suffix.lower() in extensoes_metadados
            or any(palavra in caminho.name.lower() for palavra in palavras)
        ):
            candidatos.append((item.filename, item.file_size))

    print("\nPOSSÍVEIS DOCUMENTOS E METADADOS")

    for nome, tamanho in sorted(candidatos)[:80]:
        print(f"{nome} | {tamanho:,} bytes")

print("\nInspeção concluída. Nenhum arquivo foi modificado.")

Pares confirmados: 3574

SEQUÊNCIAS NOMINAIS
Os nomes não comprovam independência espacial.

Prefixo: Langsenkamp_210722_bonirob_color
Quantidade: 1054
Primeiro e último quadro: 3679 4739
Passos mais comuns: [(1, 1052), (8, 1)]
Maiores lacunas: [(4378, 4386, 8)]

Prefixo: Langsenkamp_210727_bonirob_color
Quantidade: 671
Primeiro e último quadro: 4740 5459
Passos mais comuns: [(1, 669), (50, 1)]
Maiores lacunas: [(5286, 5336, 50)]

Prefixo: Langsenkamp_210727_color
Quantidade: 1849
Primeiro e último quadro: 0 2718
Passos mais comuns: [(1, 1844), (2, 1), (44, 1), (68, 1), (760, 1)]
Maiores lacunas: [(1859, 2619, 760), (1718, 1786, 68), (1502, 1546, 44), (618, 620, 2)]

Nomes não interpretados: []

POSSÍVEIS DOCUMENTOS E METADADOS
maize_dataset/README.txt | 326 bytes
maize_dataset/coco_annotations/train_2022.json | 32,783,294 bytes
maize_dataset/coco_annotations/valid_2022.json | 8,874,168 bytes
maize_dataset/data.yaml | 356 bytes
maize_dataset/test.txt | 244,073 bytes
maize_dataset/test_

## Auditoria de duplicatas e similaridade visual

Vamos examinar as 3.574 imagens anotadas do CornWeed, preservando os originais.

Calculamos hashes dos arquivos e dos pixels para encontrar duplicatas exatas. Também usamos hash perceptual para selecionar imagens parecidas, incluindo comparações entre sequências e datas.

Os pares selecionados serão exibidos para revisão visual. Similaridade não comprova duplicação, e a ausência de pares semelhantes não garante independência espacial.

Esta etapa não separa os conjuntos nem inicia treinamento.

In [ ]:
from pathlib import Path, PurePosixPath
from collections import defaultdict, Counter
from datetime import datetime, timezone
import hashlib
import io
import json
import re
import zipfile

import numpy as np
from PIL import Image, ImageOps
from scipy.fft import dctn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

caminho_zip = Path("/content/cornweed/maize_dataset.zip")
assert caminho_zip.is_file(), f"ZIP não encontrado: {caminho_zip}"

saida = Path("/content") / (
    "cornweed_auditoria_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%f")
)
saida.mkdir()

registros = []
miniaturas = []
hashes = []

with zipfile.ZipFile(caminho_zip) as arquivo:
    imagens = sorted(
        nome for nome in arquivo.namelist()
        if PurePosixPath(nome).parent
        == PurePosixPath("maize_dataset/data")
        and PurePosixPath(nome).suffix.lower() in {".png", ".jpg", ".jpeg"}
    )
    assert len(imagens) == 3574, len(imagens)

    for nome in tqdm(imagens, desc="Lendo imagens"):
        conteudo = arquivo.read(nome)

        with Image.open(io.BytesIO(conteudo)) as original:
            rgb = ImageOps.exif_transpose(original).convert("RGB")

        stem = PurePosixPath(nome).stem
        prefixo, quadro = stem.rsplit("_", 1)
        data = re.search(r"_(\d{6})(?:_|$)", prefixo).group(1)

        hash_pixels = hashlib.sha256(
            f"RGB:{rgb.size}:".encode() + rgb.tobytes()
        ).hexdigest()

        cinza = rgb.convert("L").resize(
            (32, 32), Image.Resampling.LANCZOS
        )
        frequencias = dctn(
            np.asarray(cinza, dtype=np.float32),
            type=2,
            norm="ortho",
        )[:8, :8].ravel()[1:]

        bits = frequencias > np.median(frequencias)
        phash = np.packbits(bits)

        registros.append({
            "imagem": nome,
            "prefixo": prefixo,
            "data": data,
            "quadro": int(quadro),
            "resolucao": list(rgb.size),
            "sha256_arquivo": hashlib.sha256(conteudo).hexdigest(),
            "sha256_pixels": hash_pixels,
            "phash": phash.tobytes().hex(),
        })
        hashes.append(phash)

        rgb.thumbnail((256, 192))
        miniaturas.append(rgb)

# Identificação de duplicatas exatas.
duplicatas = {}

for criterio in ["sha256_arquivo", "sha256_pixels"]:
    grupos = defaultdict(list)

    for registro in registros:
        grupos[registro[criterio]].append(registro["imagem"])

    duplicatas[criterio] = [
        nomes for nomes in grupos.values() if len(nomes) > 1
    ]

# Comparação perceptual de todos os pares.
H = np.stack(hashes)
popcount = np.array(
    [numero.bit_count() for numero in range(256)],
    dtype=np.uint8,
)

categorias = ["global", "entre_prefixos", "entre_datas"]
melhores = {categoria: [] for categoria in categorias}
contagens = Counter()
limiar = 10

prefixos = np.array([r["prefixo"] for r in registros])
datas = np.array([r["data"] for r in registros])

for i in tqdm(range(len(H) - 1), desc="Comparando pares"):
    distancias = popcount[
        np.bitwise_xor(H[i + 1:], H[i])
    ].sum(axis=1)

    indices = np.arange(i + 1, len(H))
    mascaras = {
        "global": np.ones(len(indices), dtype=bool),
        "entre_prefixos": prefixos[i + 1:] != prefixos[i],
        "entre_datas": datas[i + 1:] != datas[i],
    }

    for categoria, mascara in mascaras.items():
        contagens[categoria] += int(
            np.count_nonzero(mascara & (distancias <= limiar))
        )

        candidatos = np.flatnonzero(mascara)
        if candidatos.size:
            ordem = np.argsort(
                distancias[candidatos], kind="stable"
            )[:20]

            melhores[categoria].extend(
                (int(distancias[k]), i, int(indices[k]))
                for k in candidatos[ordem]
            )
            melhores[categoria] = sorted(
                melhores[categoria]
            )[:20]

# Salva inventário e resultados, sem alterar o dataset.
(saida / "inventario.json").write_text(
    json.dumps(registros, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(saida / "duplicatas_exatas.json").write_text(
    json.dumps(duplicatas, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

pares_salvos = {
    categoria: [
        {
            "imagem_a": registros[i]["imagem"],
            "imagem_b": registros[j]["imagem"],
            "distancia_phash": distancia,
            "revisao_visual": "pendente",
        }
        for distancia, i, j in pares
    ]
    for categoria, pares in melhores.items()
}

(saida / "pares_candidatos.json").write_text(
    json.dumps(pares_salvos, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Mostra quatro pares de cada categoria.
for categoria in categorias:
    pares = melhores[categoria][:4]
    if not pares:
        continue

    fig, eixos = plt.subplots(
        len(pares), 2,
        figsize=(12, 3.5 * len(pares)),
        squeeze=False,
    )

    for linha, (distancia, i, j) in enumerate(pares):
        for coluna, indice in enumerate([i, j]):
            eixo = eixos[linha, coluna]
            eixo.imshow(miniaturas[indice])
            eixo.set_title(
                f"{categoria} | distância: {distancia}\n"
                f"{PurePosixPath(registros[indice]['imagem']).name}",
                fontsize=9,
            )
            eixo.axis("off")

    plt.tight_layout()
    fig.savefig(saida / f"painel_{categoria}.png", dpi=140)
    plt.show()
    plt.close(fig)

resumo = {
    "imagens": len(registros),
    "pares_comparados": len(H) * (len(H) - 1) // 2,
    "grupos_duplicados": {
        criterio: len(grupos)
        for criterio, grupos in duplicatas.items()
    },
    "pares_com_distancia_ate_10": dict(contagens),
    "independencia_espacial": "nao_comprovada",
    "divisao": "ainda_nao_definida",
}

(saida / "resumo.json").write_text(
    json.dumps(resumo, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(resumo, indent=2, ensure_ascii=False))
print("\nRelatórios salvos em:", saida)
print("Nenhum arquivo original foi modificado.")

{
  "imagens": 3574,
  "pares_comparados": 6384951,
  "grupos_duplicados": {
    "sha256_arquivo": 0,
    "sha256_pixels": 0
  },
  "pares_com_distancia_ate_10": {
    "global": 23216,
    "entre_prefixos": 0,
    "entre_datas": 0
  },
  "independencia_espacial": "nao_comprovada",
  "divisao": "ainda_nao_definida"
}

Relatórios salvos em: /content/cornweed_auditoria_20261031T211729_613949
Nenhum arquivo original foi modificado.


## Proposta de treino, validação e teste

Reservamos as 1.054 imagens de 210722 para teste.

Em cada sequência de 210727, os primeiros 80% dos quadros formam o
treino candidato e os restantes formam a validação candidata.

Verificamos pares visualmente semelhantes que cruzam essa fronteira,
reutilizando os hashes já calculados. A divisão ainda não é definitiva
e não comprova independência espacial.

Nenhuma imagem será copiada, descartada ou usada para treinamento.

In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv
import json
import numpy as np

# Reutiliza os dados da auditoria anterior.
assert len(registros) == len(H) == 3574
hashes_bytes = np.asarray(H, dtype=np.uint8)
assert hashes_bytes.shape == (3574, 8)

itens = []
grupos = defaultdict(list)

for indice, registro in enumerate(registros):
    nome = registro["imagem"]
    prefixo, numero = PurePosixPath(nome).stem.rsplit("_", 1)
    data = registro["data"]

    assert data in {"210722", "210727"}, data

    item = {
        "indice": indice,
        "imagem": nome,
        "prefixo": prefixo,
        "data": data,
        "quadro": int(numero),
        "split": "test" if data == "210722" else "",
    }
    itens.append(item)
    grupos[prefixo].append(item)

blocos = []

for prefixo, grupo in sorted(grupos.items()):
    grupo.sort(key=lambda item: item["quadro"])

    if grupo[0]["data"] == "210722":
        partes = [("test", grupo)]
    else:
        corte = len(grupo) * 4 // 5
        partes = [
            ("train", grupo[:corte]),
            ("val", grupo[corte:]),
        ]

    for split, parte in partes:
        for item in parte:
            item["split"] = split

        blocos.append({
            "prefixo": prefixo,
            "split": split,
            "imagens": len(parte),
            "primeiro_quadro": parte[0]["quadro"],
            "ultimo_quadro": parte[-1]["quadro"],
        })

# Procura semelhanças somente entre treino e validação.
# O teste não participa da escolha das fronteiras.
indices_train = np.array([
    item["indice"] for item in itens if item["split"] == "train"
])
indices_val = np.array([
    item["indice"] for item in itens if item["split"] == "val"
])

popcount_local = np.array(
    [numero.bit_count() for numero in range(256)],
    dtype=np.uint8,
)

pares_cruzados = []
mais_proximos = []

def registrar_par(i, j, distancia):
    return {
        "imagem_train": itens[i]["imagem"],
        "imagem_val": itens[j]["imagem"],
        "distancia_phash": int(distancia),
        "prefixo_train": itens[i]["prefixo"],
        "prefixo_val": itens[j]["prefixo"],
    }

for i in indices_train:
    distancias = popcount_local[
        np.bitwise_xor(hashes_bytes[indices_val], hashes_bytes[i])
    ].sum(axis=1)

    for posicao in np.flatnonzero(distancias <= 10):
        j = int(indices_val[posicao])
        pares_cruzados.append(
            registrar_par(int(i), j, distancias[posicao])
        )

    for posicao in np.argsort(distancias, kind="stable")[:20]:
        mais_proximos.append(
            (int(distancias[posicao]), int(i), int(indices_val[posicao]))
        )

    mais_proximos = sorted(mais_proximos)[:20]

pasta_blocos = Path("/content") / (
    "cornweed_blocos_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%f")
)
pasta_blocos.mkdir()

def salvar_csv(nome, linhas, campos):
    with (pasta_blocos / nome).open(
        "w", newline="", encoding="utf-8"
    ) as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=campos)
        escritor.writeheader()
        escritor.writerows(linhas)

salvar_csv("manifesto_candidato.csv", itens, list(itens[0]))
salvar_csv("blocos_candidatos.csv", blocos, list(blocos[0]))

campos_pares = [
    "imagem_train", "imagem_val", "distancia_phash",
    "prefixo_train", "prefixo_val",
]
salvar_csv(
    "pares_train_val_ate10.csv",
    pares_cruzados,
    campos_pares,
)
salvar_csv(
    "top20_train_val.csv",
    [registrar_par(i, j, d) for d, i, j in mais_proximos],
    campos_pares,
)

resumo_blocos = {
    "status": "divisao_candidata",
    "imagens_por_divisao": dict(Counter(
        item["split"] for item in itens
    )),
    "pares_train_val_com_phash_ate10": len(pares_cruzados),
    "imagens_train_envolvidas": len({
        par["imagem_train"] for par in pares_cruzados
    }),
    "imagens_val_envolvidas": len({
        par["imagem_val"] for par in pares_cruzados
    }),
    "quadros_excluidos": 0,
    "independencia_espacial": "nao_comprovada",
    "licenca": "pendente",
}

(pasta_blocos / "resumo.json").write_text(
    json.dumps(resumo_blocos, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("BLOCOS CANDIDATOS:")
for bloco in blocos:
    print(
        bloco["prefixo"],
        "|", bloco["split"],
        "|", bloco["imagens"], "imagens",
        "| quadros", bloco["primeiro_quadro"],
        "a", bloco["ultimo_quadro"],
    )

print("\nRESUMO:")
print(json.dumps(resumo_blocos, indent=2, ensure_ascii=False))
print("\nRelatórios:", pasta_blocos)
print("Nenhuma imagem modificada. Nenhum treinamento iniciado.")

BLOCOS CANDIDATOS:
Langsenkamp_210722_bonirob_color | test | 1054 imagens | quadros 3679 a 4739
Langsenkamp_210727_bonirob_color | train | 536 imagens | quadros 4740 a 5275
Langsenkamp_210727_bonirob_color | val | 135 imagens | quadros 5276 a 5459
Langsenkamp_210727_color | train | 1479 imagens | quadros 0 a 1479
Langsenkamp_210727_color | val | 370 imagens | quadros 1480 a 2718

RESUMO:
{
  "status": "divisao_candidata",
  "imagens_por_divisao": {
    "test": 1054,
    "train": 2015,
    "val": 505
  },
  "pares_train_val_com_phash_ate10": 4,
  "imagens_train_envolvidas": 4,
  "imagens_val_envolvidas": 4,
  "quadros_excluidos": 0,
  "independencia_espacial": "nao_comprovada",
  "licenca": "pendente"
}

Relatórios: /content/cornweed_blocos_20261031T213104_864793
Nenhuma imagem modificada. Nenhum treinamento iniciado.


In [ ]:
from pathlib import Path
import csv
import io
import zipfile
import matplotlib.pyplot as plt
from PIL import Image

pasta = pasta_blocos

with (pasta / "pares_train_val_ate10.csv").open(
    encoding="utf-8", newline=""
) as f:
    pares = list(csv.DictReader(f))

print("Pares para conferir:", len(pares))

with zipfile.ZipFile(zip_cornweed) as z:
    for par in pares:
        fig, eixos = plt.subplots(1, 2, figsize=(12, 5))

        for eixo, campo, titulo in [
            (eixos[0], "imagem_train", "TREINO"),
            (eixos[1], "imagem_val", "VALIDAÇÃO"),
        ]:
            nome = par[campo]

            with Image.open(io.BytesIO(z.read(nome))) as original:
                imagem = original.convert("RGB")

            eixo.imshow(imagem)
            eixo.set_title(f"{titulo}\n{Path(nome).name}", fontsize=9)
            eixo.axis("off")

        fig.suptitle(
            f"Distância pHash: {par['distancia_phash']}"
        )
        plt.tight_layout()
        plt.show()

print("Somente visualização. Nenhum arquivo foi alterado.")

## Preparação do CornWeed para o segundo experimento

Divisão exploratória:
- Treino: 2.015 imagens.
- Validação: 505 imagens.
- Teste reservado: 1.054 imagens da aquisição de 210722.

As aquisições são do mesmo campo; independência espacial não comprovada.
O teste ficará fora da escolha dos parâmetros de treinamento.

As imagens serão copiadas sem alteração. Os rótulos serão convertidos
somente na cópia: Weeds (0) → weed (1); Maize (1) → crop (0).

O ZIP original e o primeiro experimento serão preservados.
A licença permanece pendente de esclarecimento.
Esta etapa não executa treinamento.

In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone
import csv
import hashlib
import json
import zipfile
from tqdm.auto import tqdm

origem_zip = Path("/content/cornweed/maize_dataset.zip")
origem_manifesto = pasta_blocos / "manifesto_candidato.csv"

assert origem_zip.is_file(), "ZIP não encontrado."
assert origem_manifesto.is_file(), "Manifesto não encontrado."

with origem_manifesto.open(encoding="utf-8", newline="") as f:
    linhas = list(csv.DictReader(f))

esperado = {"train": 2015, "val": 505, "test": 1054}
assert dict(Counter(l["split"] for l in linhas)) == esperado
assert len({l["imagem"] for l in linhas}) == 3574

destino_cornweed = Path(
    "/content/cornweed_preparado_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%f")
)
destino_cornweed.mkdir(exist_ok=False)

for split in esperado:
    (destino_cornweed / "images" / split).mkdir(parents=True)
    (destino_cornweed / "labels" / split).mkdir(parents=True)

manifesto_final = []
caixas = {split: Counter() for split in esperado}

with zipfile.ZipFile(origem_zip) as z:
    for linha in tqdm(linhas, desc="Preparando imagens e rótulos"):
        imagem = PurePosixPath(linha["imagem"])
        split = linha["split"]

        assert imagem.parent == PurePosixPath("maize_dataset/data")
        rotulo = imagem.with_suffix(".txt")

        conteudo_imagem = z.read(str(imagem))
        texto_original = z.read(str(rotulo)).decode("utf-8-sig")

        novas_linhas = []
        for anotacao in texto_original.splitlines():
            if not anotacao.strip():
                continue

            campos = anotacao.split()
            assert len(campos) == 5, f"Rótulo inválido: {rotulo}"
            classe_original = int(campos[0])
            assert classe_original in (0, 1)

            classe_nova = {0: 1, 1: 0}[classe_original]
            novas_linhas.append(
                f"{classe_nova} " + " ".join(campos[1:])
            )
            caixas[split][classe_nova] += 1

        assert novas_linhas, f"Rótulo vazio: {rotulo}"
        conteudo_rotulo = (
            "\n".join(novas_linhas) + "\n"
        ).encode("utf-8")

        imagem_saida = destino_cornweed / "images" / split / imagem.name
        rotulo_saida = destino_cornweed / "labels" / split / rotulo.name

        with imagem_saida.open("xb") as f:
            f.write(conteudo_imagem)
        with rotulo_saida.open("xb") as f:
            f.write(conteudo_rotulo)

        manifesto_final.append({
            **linha,
            "sha256_imagem": hashlib.sha256(
                conteudo_imagem
            ).hexdigest(),
            "sha256_rotulo_convertido": hashlib.sha256(
                conteudo_rotulo
            ).hexdigest(),
        })

assert sum(c[0] for c in caixas.values()) == 23985
assert sum(c[1] for c in caixas.values()) == 257740

with (destino_cornweed / "manifesto.csv").open(
    "x", encoding="utf-8", newline=""
) as f:
    escritor = csv.DictWriter(
        f, fieldnames=list(manifesto_final[0])
    )
    escritor.writeheader()
    escritor.writerows(manifesto_final)

(destino_cornweed / "data.yaml").write_text(
    f"path: {destino_cornweed.as_posix()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "names:\n"
    "  0: crop\n"
    "  1: weed\n",
    encoding="utf-8",
)

registro = {
    "imagens": esperado,
    "classes": {"0": "crop", "1": "weed"},
    "caixas_por_divisao": {
        split: {"crop": c[0], "weed": c[1]}
        for split, c in caixas.items()
    },
    "independencia_espacial": "nao_comprovada",
    "licenca": "pendente",
    "treinamento_executado": False,
}

(destino_cornweed / "preparacao.json").write_text(
    json.dumps(registro, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nPREPARAÇÃO CONCLUÍDA")
print("Pasta:", destino_cornweed)
print("Configuração:", destino_cornweed / "data.yaml")
print(json.dumps(registro, indent=2, ensure_ascii=False))
print("Originais preservados. Nenhum treinamento iniciado.")


PREPARAÇÃO CONCLUÍDA
Pasta: /content/cornweed_preparado_20261031T213748_824103
Configuração: /content/cornweed_preparado_20261031T213748_824103/data.yaml
{
  "imagens": {
    "train": 2015,
    "val": 505,
    "test": 1054
  },
  "classes": {
    "0": "crop",
    "1": "weed"
  },
  "caixas_por_divisao": {
    "train": {
      "crop": 15796,
      "weed": 164586
    },
    "val": {
      "crop": 4083,
      "weed": 41575
    },
    "test": {
      "crop": 4106,
      "weed": 51579
    }
  },
  "independencia_espacial": "nao_comprovada",
  "licenca": "pendente",
  "treinamento_executado": false
}
Originais preservados. Nenhum treinamento iniciado.


In [ ]:
import requests
import json
from pathlib import Path

consultas = {
    "zenodo": "https://zenodo.org/api/records/7961764",
    "datacite": "https://api.datacite.org/dois/10.5281/zenodo.7961764",
}

pasta = destino_cornweed

for nome, url in consultas.items():
    print(f"\n--- {nome.upper()} ---")
    try:
        resposta = requests.get(url, timeout=30)
        print("HTTP:", resposta.status_code)
        resposta.raise_for_status()
        dados = resposta.json()

        (pasta / f"metadados_{nome}.json").write_text(
            json.dumps(dados, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        if nome == "zenodo":
            metadata = dados.get("metadata", {})
            print("Licença:", metadata.get("license"))
            print("Condições de acesso:", metadata.get("access_right"))
            print("Notas:", metadata.get("access_conditions"))
        else:
            atributos = dados.get("data", {}).get("attributes", {})
            print("Direitos:", atributos.get("rightsList"))

    except Exception as erro:
        print("Não foi possível consultar:", str(erro))


--- ZENODO ---
HTTP: 200
Licença: {'id': 'cc-by-4.0'}
Condições de acesso: open
Notas: None

--- DATACITE ---
HTTP: 200
Direitos: [{'rights': 'Creative Commons Attribution 4.0 International', 'rightsUri': 'https://creativecommons.org/licenses/by/4.0/legalcode', 'schemeUri': 'https://spdx.org/licenses/', 'rightsIdentifier': 'cc-by-4.0', 'rightsIdentifierScheme': 'SPDX'}, {'rights': 'Open Access', 'rightsUri': 'info:eu-repo/semantics/openAccess'}]


## Segundo experimento — YOLOv8n com CornWeed

Fonte: https://zenodo.org/records/7961764
Licença confirmada nos metadados Zenodo e DataCite: CC BY 4.0.
A entrega deverá incluir a referência e os créditos aos autores.

Treinamento com 2.015 imagens e validação com 505 imagens.
As 1.054 imagens de teste ficarão reservadas para a avaliação final.
A independência espacial entre conjuntos não está comprovada.

Classes convertidas: 0 = crop (milho); 1 = weed.
Este experimento utiliza somente CornWeed e não comprova desempenho
em soja ou preservação do desempenho anterior em gergelim.

O modelo inicial será YOLOv8n pré-treinado. O primeiro experimento
e seus pesos serão preservados.

In [ ]:
%pip install -q ultralytics==8.4.190

In [ ]:
from pathlib import Path
import json
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), (
    "GPU indisponível. Ative a GPU no Colab antes de continuar."
)

pasta_dados = destino_cornweed
assert (pasta_dados / "data.yaml").is_file()

# Atualiza o registro com a licença confirmada.
registro_path = pasta_dados / "preparacao.json"
registro = json.loads(registro_path.read_text(encoding="utf-8"))
registro["licenca"] = "CC-BY-4.0"
registro["fonte"] = "https://zenodo.org/records/7961764"
registro_path.write_text(
    json.dumps(registro, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("GPU:", torch.cuda.get_device_name(0))

modelo_cornweed = YOLO("yolov8n.pt")

resultado_treino = modelo_cornweed.train(
    data=str(pasta_dados / "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    seed=42,
    patience=10,
    cache=False,
    project="/content/resultados_cornweed",
    name="yolov8n_30ep",
    exist_ok=False,
    plots=True,
)

print("Resultados salvos em:", modelo_cornweed.trainer.save_dir)

Validating /content/resultados_cornweed/yolov8n_30ep/weights/best.pt...
Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 1.3it/s 25.4s
                   all        505      45658       0.84      0.823      0.867      0.519
                  crop        504       4083      0.879      0.912      0.935      0.668
                  weed        505      41575      0.802      0.733        0.8      0.369
Speed: 0.7ms preprocess, 5.7ms inference, 0.0ms loss, 5.4ms postprocess per image
Results saved to /content/resultados_cornweed/yolov8n_30ep
Resultados salvos em: /content/resultados_cornweed/yolov8n_30ep


In [ ]:
from ultralytics import YOLO
import json

modelo_teste = YOLO(
    "/content/resultados_cornweed/yolov8n_30ep/weights/best.pt"
)

metricas_teste = modelo_teste.val(
    data=str(destino_cornweed / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    conf=0.001,
    iou=0.7,
    max_det=300,
    project="/content/resultados_cornweed",
    name="avaliacao_teste",
    plots=True,
)

resultados = {
    chave: float(valor)
    for chave, valor in metricas_teste.results_dict.items()
}

(metricas_teste.save_dir / "metricas_teste.json").write_text(
    json.dumps(resultados, indent=2),
    encoding="utf-8",
)

print("\nMÉTRICAS DO TESTE:")
print(json.dumps(resultados, indent=2))
print("Arquivos salvos em:", metricas_teste.save_dir)

Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 84.3±19.6 MB/s, size: 519.7 KB)
val: Scanning /content/cornweed_preparado_20261031T213748_824103/labels/test... 1054 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1054/1054 272.3it/s 3.9s
val: New cache created: /content/cornweed_preparado_20261031T213748_824103/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 132/132 5.9it/s 22.2s
                   all       1054      55685      0.822      0.833      0.833      0.501
                  crop       1053       4106      0.966      0.968      0.989      0.739
                  weed       1053      51579      0.679      0.698      0.677      0.264
Speed: 1.2ms preprocess, 4.9ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to /co

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

resultados = Path("/content/resultados_cornweed")
dados = destino_cornweed
pacote = Path("/content/Demeter_CornWeed_resultados.zip")

with ZipFile(pacote, "w", ZIP_DEFLATED) as z:
    # Pesos, gráficos, configurações e métricas.
    for arquivo in resultados.rglob("*"):
        if arquivo.is_file():
            z.write(
                arquivo,
                Path("resultados_cornweed") / arquivo.relative_to(resultados)
            )

    # Manifesto e metadados, sem incluir o dataset inteiro.
    for arquivo in dados.iterdir():
        if arquivo.is_file():
            z.write(arquivo, Path("preparacao") / arquivo.name)

    # Relatórios das auditorias.
    for padrao in ("cornweed_auditoria_*", "cornweed_blocos_*"):
        for pasta in Path("/content").glob(padrao):
            if pasta.is_dir():
                for arquivo in pasta.rglob("*"):
                    if arquivo.is_file():
                        z.write(
                            arquivo,
                            Path("auditorias") / pasta.name
                            / arquivo.relative_to(pasta)
                        )

print("Pacote:", pacote)
print("Tamanho:", round(pacote.stat().st_size / 1024**2, 1), "MB")
files.download(str(pacote))